# APAN 5200 — Ensemble Models Mock Quiz 1
**Dataset:** `spotify_data.csv` | **Outcome:** `rating` (continuous)

Same structure as Question 8. **Configuration:** `n_estimators=50`, `random_state=1031`

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    VotingRegressor, BaggingRegressor,
    RandomForestRegressor, AdaBoostRegressor,
    GradientBoostingRegressor
)
from sklearn.metrics import mean_squared_error

In [ ]:
# ============================================================
# Q1: Read spotify_data.csv. Drop: id, performer, song, genre, key.
#     y = rating, X = remaining features.
#     y_binned = pd.qcut(data.rating, q=20)
#     Stratified split: 70% train / 30% test, random_state=1031
#
# QUESTION: What is the median DANCEABILITY in the train sample?
# ANSWER  : 0.607
# ============================================================

data = pd.read_csv('spotify_data.csv')
data = data.drop(columns=['id','performer','song','genre','key'])
y = data['rating']
X = data.drop(columns=['rating'])

y_binned = pd.qcut(data.rating, q=20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1031, stratify=y_binned
)

print('Train size:', X_train.shape[0], '| Test size:', X_test.shape[0])
print('Median danceability (train):', X_train['danceability'].median())  # ✅ 0.607

In [ ]:
# ============================================================
# Q2: Construct a LINEAR REGRESSION using ALL predictors.
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 15.3494
# ============================================================

lr_all = LinearRegression()
lr_all.fit(X_train, y_train)
rmse_lr = np.sqrt(mean_squared_error(y_train, lr_all.predict(X_train)))
print(f'Linear Regression (all features) RMSE (train): {rmse_lr:.4f}')  # ✅ 15.3494

In [ ]:
# ============================================================
# Q3: Construct a TREE MODEL using ONLY genre_ dummies. Use max_depth=6.
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 15.4701
# ============================================================

genre_cols = [c for c in X.columns if c.startswith('genre_')]
tree = DecisionTreeRegressor(max_depth=6, random_state=1031)
tree.fit(X_train[genre_cols], y_train)
rmse_tree = np.sqrt(mean_squared_error(y_train, tree.predict(X_train[genre_cols])))
print(f'Tree (max_depth=6, genre) RMSE (train): {rmse_tree:.4f}')  # ✅ 15.4701

In [ ]:
# ============================================================
# Q4: Combine the two models above using a VOTING model.
# QUESTION: What is the train sample RMSE?
# ANSWER  : 15.5582
# KEY: VotingRegressor uses genre_ dummies for both base learners.
# ============================================================

voting = VotingRegressor(estimators=[
    ('lr',   LinearRegression()),
    ('tree', DecisionTreeRegressor(max_depth=6, random_state=1031))
])
voting.fit(X_train[genre_cols], y_train)
rmse_voting = np.sqrt(mean_squared_error(y_train, voting.predict(X_train[genre_cols])))
print(f'Voting RMSE (train): {rmse_voting:.4f}')  # ✅ 15.5582

In [ ]:
# ============================================================
# Q5: Construct a BAGGING model with 50 bootstrapped samples.
#     Fit tree estimators using ALL predictors. Set max_depth=5.
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 15.1055
# ============================================================

bagging_tree = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=5),
    n_estimators=50,
    random_state=1031,
    oob_score=True
)
bagging_tree.fit(X_train, y_train)
rmse_bag_tree = np.sqrt(mean_squared_error(y_train, bagging_tree.predict(X_train)))
print(f'Bagging Tree (md=5, n=50) RMSE (train): {rmse_bag_tree:.4f}')  # ✅ 15.1055
print(f'Bagging Tree OOB R²: {bagging_tree.oob_score_:.4f}')           # ✅ 0.1495

In [ ]:
# ============================================================
# Q6: Construct a BAGGING model with 50 bootstrapped samples,
#     but this time fit a LINEAR REGRESSION with ALL predictors.
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 15.3496
# ============================================================

bagging_lr = BaggingRegressor(
    estimator=LinearRegression(),
    n_estimators=50,
    random_state=1031,
    oob_score=True
)
bagging_lr.fit(X_train, y_train)
rmse_bag_lr = np.sqrt(mean_squared_error(y_train, bagging_lr.predict(X_train)))
print(f'Bagging LR (n=50) RMSE (train): {rmse_bag_lr:.4f}')  # ✅ 15.3496
print(f'Bagging LR OOB R²: {bagging_lr.oob_score_:.4f}')      # ✅ 0.1435

In [ ]:
# ============================================================
# Q7: Fit a RANDOM FOREST model with 50 trees. Use ALL predictors.
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 5.7883
# ============================================================

rf = RandomForestRegressor(n_estimators=50, random_state=1031, oob_score=True)
rf.fit(X_train, y_train)
rmse_rf = np.sqrt(mean_squared_error(y_train, rf.predict(X_train)))
print(f'Random Forest (n=50) RMSE (train): {rmse_rf:.4f}')  # ✅ 5.7883
print(f'Random Forest OOB R²: {rf.oob_score_:.4f}')          # ✅ 0.1286

In [ ]:
# ============================================================
# Q8: Based on the Random Forest above, which feature is MOST important?
# ANSWER  : track_duration (importance ≈ 0.1480)
# ============================================================

feat_imp = pd.Series(rf.feature_importances_, index=X_train.columns)\
             .sort_values(ascending=False)
print('Top 5 Feature Importances (Random Forest):')
print(feat_imp.head(5).to_string())  # ✅ track_duration top

In [ ]:
# ============================================================
# Q9: Which of the models above has the LOWEST OOB score?
# ANSWER  : Random Forest (OOB ≈ 0.1286)
#
# NOTE: Lower OOB R² = worse generalisation on OOB samples.
# RF has the lowest OOB despite the best train RMSE — it overfits
# more on the training bootstrap samples.
# ============================================================

print('OOB R² Scores:')
print(f'  Bagging Tree (md=5, n=50): {bagging_tree.oob_score_:.4f}')
print(f'  Bagging LR   (n=50)      : {bagging_lr.oob_score_:.4f}')
print(f'  Random Forest (n=50)     : {rf.oob_score_:.4f}  ← LOWEST')

In [ ]:
# ============================================================
# Q10: Construct a BOOSTING model using ADABOOST with a tree
#      estimator. Use 50 estimators. Use ALL predictors.
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 0.5897
# NOTE: Near-zero train RMSE → severe overfitting on train set!
# ============================================================

ada = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(),
    n_estimators=50,
    random_state=1031
)
ada.fit(X_train, y_train)
rmse_ada = np.sqrt(mean_squared_error(y_train, ada.predict(X_train)))
print(f'AdaBoost (n=50) RMSE (train): {rmse_ada:.4f}')  # ✅ 0.5897

In [ ]:
# ============================================================
# Q11: Construct a GRADIENT BOOSTING model with 50 trees.
#      Use ALL predictors.
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 14.9630
# ============================================================

gb = GradientBoostingRegressor(n_estimators=50, random_state=1031)
gb.fit(X_train, y_train)
rmse_gb = np.sqrt(mean_squared_error(y_train, gb.predict(X_train)))
print(f'Gradient Boosting (n=50) RMSE (train): {rmse_gb:.4f}')  # ✅ 14.9630

In [ ]:
# ============================================================
# Q12: Construct a STOCHASTIC GRADIENT BOOSTING model with 50 trees.
#      Set max_features=0.5 and subsample=0.7. Use ALL predictors.
# QUESTION: What is the RMSE in the train sample?
# ANSWER  : 15.0078
# ============================================================

sgb = GradientBoostingRegressor(
    n_estimators=50,
    max_features=0.5,
    subsample=0.7,
    random_state=1031
)
sgb.fit(X_train, y_train)
rmse_sgb = np.sqrt(mean_squared_error(y_train, sgb.predict(X_train)))
print(f'Stochastic GB (n=50, mf=0.5, sub=0.7) RMSE (train): {rmse_sgb:.4f}')  # ✅ 15.0078

In [ ]:
# ============================================================
# Q13: Based on the Stochastic GB above, which is the most important feature?
# ANSWER: track_duration (importance ≈ 0.2655)
# ============================================================

feat_imp_sgb = pd.Series(sgb.feature_importances_, index=X_train.columns)\
                 .sort_values(ascending=False)
print('Top 5 Feature Importances (Stochastic GB):')
print(feat_imp_sgb.head(5).to_string())  # ✅ track_duration top

---
## Answer Summary

In [ ]:
summary = {
    'Q1  Median danceability (train)':         '0.607',
    'Q2  LR (all features) RMSE train':        '15.3494',
    'Q3  Tree (md=6, genre) RMSE train':       '15.4701',
    'Q4  Voting RMSE train':                    '15.5582',
    'Q5  Bagging Tree (md=5, n=50) RMSE':      '15.1055',
    'Q6  Bagging LR (n=50) RMSE':              '15.3496',
    'Q7  Random Forest (n=50) RMSE':           '5.7883',
    'Q8  RF most important feature':            'track_duration',
    'Q9  Lowest OOB score model':               'Random Forest (0.1286)',
    'Q10 AdaBoost (n=50) RMSE':                '0.5897',
    'Q11 Gradient Boosting (n=50) RMSE':       '14.9630',
    'Q12 Stochastic GB RMSE':                  '15.0078',
    'Q13 SGB most important feature':           'track_duration',
}
for k,v in summary.items():
    print(f'{k:<45} → {v}')